# UNISOLAR Dataset Exploration

## Project
Solar Power Forecasting using Hybrid XGBoost-LSTM

## Dataset
UNISOLAR Solar Power Generation Dataset

## Mục tiêu
Notebook này thực hiện phân tích dữ liệu khám phá (EDA) trên bộ dữ liệu UNISOLAR
nhằm phục vụ việc dự báo ngắn hạn sản lượng điện mặt trời.


## 1. Import thư viện

In [1]:
import os
import glob
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

pd.set_option("display.max_columns", None)

## 2. Load file

In [2]:
def find_file(filename):
    files = glob.glob(
        f"/kaggle/input/**/{filename}",
        recursive=True
    )
    
    if len(files) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy file: {filename}"
        )
    
    print(filename, "->", files[0])
    return files[0]


solar_path = find_file(
    "Solar_Energy_Generation.csv"
)

site_path = find_file(
    "Solar_Site_Details.csv"
)

monthly_path = find_file(
    "Monthly_Summary_Solar.csv"
)

irradiance_path = find_file(
    "Solar_Irradiance.csv"
)

Solar_Energy_Generation.csv -> /kaggle/input/datasets/cdaclab/unisolar/unisolar/Solar_Energy_Generation.csv
Solar_Site_Details.csv -> /kaggle/input/datasets/cdaclab/unisolar/unisolar/Solar_Site_Details.csv
Monthly_Summary_Solar.csv -> /kaggle/input/datasets/cdaclab/unisolar/unisolar/Monthly_Summary_Solar.csv
Solar_Irradiance.csv -> /kaggle/input/datasets/maiemm/solar-irradiance/Solar_Irradiance.csv


In [3]:
weather_files = glob.glob(
    "/kaggle/input/**/Weather_Data_*all.csv",
    recursive=True
)

print(weather_files)

weather_path = weather_files[0]

['/kaggle/input/datasets/cdaclab/unisolar/unisolar/Weather_Data_reordered_all.csv']


## 3. Đọc dữ liệu

In [4]:
solar = pd.read_csv(solar_path)

weather = pd.read_csv(weather_path)

site = pd.read_csv(site_path)

monthly = pd.read_csv(monthly_path)

irradiance = pd.read_csv(irradiance_path)

In [5]:
print("Solar:", solar.shape)
print("Weather:", weather.shape)
print("Site:", site.shape)
print("Monthly:", monthly.shape)
print("Irradiance:", irradiance.shape)

Solar: (2731946, 4)
Weather: (371769, 8)
Site: (42, 10)
Monthly: (1176, 7)
Irradiance: (102360, 4)


## 4. Xem cấu trúc dữ liệu

In [6]:
datasets = {
    "Solar": solar,
    "Weather": weather,
    "Site": site,
    "Monthly": monthly,
    "Irradiance": irradiance
}

for name, df in datasets.items():
    print("\n========================")
    print(name)
    print("========================")
    
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())
    
    display(df.head())


Solar
Shape: (2731946, 4)
Columns:
['CampusKey', 'SiteKey', 'Timestamp', 'SolarGeneration']


,CampusKey,SiteKey,Timestamp,SolarGeneration
0,2,1,2020-01-01 00:15:00,NaN
1,2,1,2020-01-01 00:30:00,NaN
2,2,1,2020-01-01 00:45:00,NaN
3,2,1,2020-01-01 01:00:00,NaN
4,2,1,2020-01-01 01:15:00,NaN



Weather
Shape: (371769, 8)
Columns:
['CampusKey', 'Timestamp', 'ApparentTemperature', 'AirTemperature', 'DewPointTemperature', 'RelativeHumidity', 'WindSpeed', 'WindDirection']


,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
0,1,2020-01-01 00:00:00,13.666667,13.880000,8.960000,72.400000,0.000000,188.133333
1,1,2020-01-01 00:15:00,13.206667,13.666667,9.040000,73.466667,1.200000,203.866667
2,1,2020-01-01 00:30:00,12.840000,13.553333,9.053333,74.000000,2.520000,222.800000
3,1,2020-01-01 00:45:00,12.113333,13.506667,9.100000,74.466667,5.986667,231.133333
4,1,2020-01-01 01:00:00,11.946667,13.260000,9.266667,76.533333,5.946667,247.866667



Site
Shape: (42, 10)
Columns:
['CampusKey', 'SiteKey', 'kWp', 'Number of panels', 'Panel', 'Inverter', 'Optimizers', 'Metric', 'lat', 'Lon']


,CampusKey,SiteKey,kWp,Number of panels,Panel,Inverter,Optimizers,Metric,lat,Lon
0,2,1,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
1,2,2,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
2,2,3,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
3,2,4,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
4,2,5,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679



Monthly
Shape: (1176, 7)
Columns:
['SiteKey', 'Year', 'Month', 'DataStatus', 'AverageSolarGeneration', 'MaxSolarGeneration', 'MinSolarGeneration']


,SiteKey,Year,Month,DataStatus,AverageSolarGeneration,MaxSolarGeneration,MinSolarGeneration
0,1,2020,January,True,10.157125,21.778,0.101
1,2,2020,January,True,8.536875,18.553,0.100
2,3,2020,January,True,6.309834,13.322,0.103
3,4,2020,January,False,NaN,NaN,NaN
4,5,2020,January,False,NaN,NaN,NaN



Irradiance
Shape: (102360, 4)
Columns:
['CloudOpacity', 'Ghi', 'Timestamp_UTC', 'Campus']


,CloudOpacity,Ghi,Timestamp_UTC,Campus
0,15.5,823,2019-12-31 00:00:00,Bundoora
1,0.0,1063,2019-12-31 01:00:00,Bundoora
2,0.0,1090,2019-12-31 02:00:00,Bundoora
3,0.0,1049,2019-12-31 03:00:00,Bundoora
4,0.0,948,2019-12-31 04:00:00,Bundoora


## 5. Chuyển timestamp

In [7]:
#solar and weather
solar["Timestamp"] = pd.to_datetime(
    solar["Timestamp"],
    errors="coerce"
)

weather["Timestamp"] = pd.to_datetime(
    weather["Timestamp"],
    errors="coerce"
)

In [8]:
#irradiance
irradiance["Timestamp_UTC"] = pd.to_datetime(
    irradiance["Timestamp_UTC"],
    errors="coerce",
    utc=True
)
irradiance["Timestamp"] = (
    irradiance["Timestamp_UTC"]
    .dt.tz_convert("Australia/Melbourne")
    .dt.tz_localize(None)
)
irradiance[
    ["Timestamp_UTC", "Timestamp", "Ghi"]
].head()

,Timestamp_UTC,Timestamp,Ghi
0,2019-12-31 00:00:00+00:00,2019-12-31 11:00:00,823
1,2019-12-31 01:00:00+00:00,2019-12-31 12:00:00,1063
2,2019-12-31 02:00:00+00:00,2019-12-31 13:00:00,1090
3,2019-12-31 03:00:00+00:00,2019-12-31 14:00:00,1049
4,2019-12-31 04:00:00+00:00,2019-12-31 15:00:00,948


## 6. Kiểm tra missing

In [11]:
def missing_summary(df):
    
    result = pd.DataFrame({
        "Missing": df.isna().sum(),
        "Missing_Percent":
            df.isna().mean() * 100
    })
    
    return result.sort_values(
        "Missing_Percent",
        ascending=False
    )


display(missing_summary(solar))
display(missing_summary(weather))
display(missing_summary(irradiance))
display(missing_summary(site))
display(missing_summary(monthly))

,Missing,Missing_Percent
SolarGeneration,1536301,56.234677
CampusKey,0,0.000000
SiteKey,0,0.000000
Timestamp,0,0.000000


,Missing,Missing_Percent
WindSpeed,162890,43.814842
WindDirection,162890,43.814842
AirTemperature,107113,28.811708
ApparentTemperature,107113,28.811708
RelativeHumidity,107113,28.811708
DewPointTemperature,107113,28.811708
CampusKey,0,0.000000
Timestamp,0,0.000000


,Missing,Missing_Percent
CloudOpacity,0,0.0
Ghi,0,0.0
Timestamp_UTC,0,0.0
Campus,0,0.0
Timestamp,0,0.0


,Missing,Missing_Percent
Optimizers,19,45.238095
kWp,17,40.476190
Inverter,17,40.476190
Metric,17,40.476190
Panel,17,40.476190
Number of panels,17,40.476190
SiteKey,0,0.000000
CampusKey,0,0.000000
lat,0,0.000000
Lon,0,0.000000


,Missing,Missing_Percent
MinSolarGeneration,200,17.006803
MaxSolarGeneration,200,17.006803
AverageSolarGeneration,200,17.006803
Month,0,0.000000
Year,0,0.000000
SiteKey,0,0.000000
DataStatus,0,0.000000


## 7. kiểm tra duplicate

In [12]:
# 1. Solar Generation
print("Solar duplicate:")
print(
    solar.duplicated(
        subset=["CampusKey", "SiteKey", "Timestamp"]
    ).sum()
)

# 2. Weather
print("Weather duplicate:")
print(
    weather.duplicated(
        subset=["CampusKey", "Timestamp"]
    ).sum()
)

# 3. Solar Site Details
print("Site duplicate:")
print(
    site.duplicated(
        subset=["CampusKey", "SiteKey"]
    ).sum()
)

# 4. Monthly Summary
print("Monthly duplicate:")
print(
    monthly.duplicated(
        subset=["SiteKey", "Year", "Month"]
    ).sum()
)

# 5. Solar Irradiance
print("Irradiance duplicate:")
print(
    irradiance.duplicated(
        subset=["Campus", "Timestamp_UTC"]
    ).sum()
)

Solar duplicate:
0
Weather duplicate:
0
Site duplicate:
0
Monthly duplicate:
0
Irradiance duplicate:
0
